In [1]:
# ============================================
# RETAINIQ - FEATURE ENGINEERING
# ============================================

import pandas as pd
import numpy as np

# Load dataset
df = pd.read_csv("../data/WA_Fn-UseC_-Telco-Customer-Churn.csv")

# Convert TotalCharges to numeric
df["TotalCharges"] = pd.to_numeric(
    df["TotalCharges"],
    errors="coerce"
)

# Remove invalid TotalCharges rows
df.dropna(subset=["TotalCharges"], inplace=True)

# Remove duplicate rows
df.drop_duplicates(inplace=True)

# Remove customer identifier
df.drop(columns=["customerID"], inplace=True)

# Convert target variable
df["Churn"] = df["Churn"].map({
    "Yes": 1,
    "No": 0
})

print("Dataset prepared successfully.")
print("Shape:", df.shape)

display(df.head())

Dataset prepared successfully.
Shape: (7032, 20)


,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,0
1,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,0
2,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,1
3,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,0
4,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,1


In [2]:
# ============================================
# SEPARATE FEATURES AND TARGET
# ============================================

X = df.drop("Churn", axis=1)
y = df["Churn"]

print("Features shape:", X.shape)
print("Target shape:", y.shape)

print("\nTarget distribution:")
print(y.value_counts())

Features shape: (7032, 19)
Target shape: (7032,)

Target distribution:
Churn
0    5163
1    1869
Name: count, dtype: int64


In [3]:
# ============================================
# IDENTIFY NUMERICAL AND CATEGORICAL FEATURES
# ============================================

numerical_features = X.select_dtypes(
    include=np.number
).columns.tolist()

categorical_features = X.select_dtypes(
    exclude=np.number
).columns.tolist()

print("Numerical Features:")
print(numerical_features)

print("\nCategorical Features:")
print(categorical_features)

print("\nNumber of numerical features:", len(numerical_features))
print("Number of categorical features:", len(categorical_features))

Numerical Features:
['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges']

Categorical Features:
['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod']

Number of numerical features: 4
Number of categorical features: 15


In [4]:
# ============================================
# IDENTIFY NUMERICAL AND CATEGORICAL FEATURES
# ============================================

numerical_features = X.select_dtypes(
    include=np.number
).columns.tolist()

categorical_features = X.select_dtypes(
    exclude=np.number
).columns.tolist()

print("Numerical Features:")
print(numerical_features)

print("\nCategorical Features:")
print(categorical_features)

print("\nNumber of numerical features:", len(numerical_features))
print("Number of categorical features:", len(categorical_features))

Numerical Features:
['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges']

Categorical Features:
['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod']

Number of numerical features: 4
Number of categorical features: 15


In [5]:
# ============================================
# PREPROCESSING PIPELINE
# ============================================

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

categorical_transformer = OneHotEncoder(
    handle_unknown="ignore",
    drop="first"
)

numerical_transformer = StandardScaler()

preprocessor_scaled = ColumnTransformer(
    transformers=[
        ("categorical", categorical_transformer, categorical_features),
        ("numerical", numerical_transformer, numerical_features)
    ]
)

print("Scaled preprocessing pipeline created successfully.")

Scaled preprocessing pipeline created successfully.


In [7]:
# ============================================
# APPLY PREPROCESSING
# ============================================

X_processed = preprocessor_scaled.fit_transform(X)

print("Original feature shape:", X.shape)
print("Processed feature shape:", X_processed.shape)

Original feature shape: (7032, 19)
Processed feature shape: (7032, 30)


In [8]:
# ============================================
# TRAIN / TEST SPLIT
# ============================================

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training features:", X_train.shape)
print("Testing features:", X_test.shape)

print("\nTraining target distribution:")
print(y_train.value_counts(normalize=True).round(3))

print("\nTesting target distribution:")
print(y_test.value_counts(normalize=True).round(3))

Training features: (5625, 19)
Testing features: (1407, 19)

Training target distribution:
Churn
0    0.734
1    0.266
Name: proportion, dtype: float64

Testing target distribution:
Churn
0    0.734
1    0.266
Name: proportion, dtype: float64


In [9]:
# ============================================
# LOGISTIC REGRESSION PIPELINE
# ============================================

from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

logistic_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor_scaled),
        ("model", LogisticRegression(
            solver="liblinear",
            max_iter=1000,
            random_state=42
        ))
    ]
)

logistic_pipeline.fit(X_train, y_train)

print("Logistic Regression model trained successfully.")

Logistic Regression model trained successfully.


In [10]:
# ============================================
# LOGISTIC REGRESSION PREDICTIONS
# ============================================

# Predict class labels
y_pred = logistic_pipeline.predict(X_test)

# Predict churn probabilities
y_pred_proba = logistic_pipeline.predict_proba(X_test)[:, 1]

print("Predictions generated successfully.")
print("First 10 predicted classes:", y_pred[:10])
print("First 10 churn probabilities:", y_pred_proba[:10])

Predictions generated successfully.
First 10 predicted classes: [0 1 0 0 0 0 0 0 1 0]
First 10 churn probabilities: [0.01819622 0.58988756 0.00494765 0.20233596 0.10447791 0.47371948
 0.02623306 0.16603513 0.67068106 0.0160187 ]


In [11]:
# ============================================
# LOGISTIC REGRESSION EVALUATION
# ============================================

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

# Calculate evaluation metrics
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_pred_proba)

print("Logistic Regression Performance")
print("=" * 40)
print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1-Score : {f1:.4f}")
print(f"ROC-AUC  : {roc_auc:.4f}")

Logistic Regression Performance
Accuracy : 0.8031
Precision: 0.6474
Recall   : 0.5695
F1-Score : 0.6060
ROC-AUC  : 0.8362


In [12]:
# ============================================
# CONFUSION MATRIX
# ============================================

cm = confusion_matrix(y_test, y_pred)

print("Confusion Matrix:")
print(cm)

print("\nClassification Report:")
print(classification_report(
    y_test,
    y_pred,
    target_names=["No Churn", "Churn"]
))

Confusion Matrix:
[[917 116]
 [161 213]]

Classification Report:
              precision    recall  f1-score   support

    No Churn       0.85      0.89      0.87      1033
       Churn       0.65      0.57      0.61       374

    accuracy                           0.80      1407
   macro avg       0.75      0.73      0.74      1407
weighted avg       0.80      0.80      0.80      1407



In [14]:
# ============================================
# RANDOM FOREST PIPELINE
# ============================================

from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier

random_forest_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor_scaled),
        ("model", RandomForestClassifier(
            n_estimators=300,
            random_state=42,
            class_weight="balanced",
            n_jobs=-1
        ))
    ]
)

random_forest_pipeline.fit(X_train, y_train)

print("Random Forest model trained successfully.")

Random Forest model trained successfully.


In [15]:
# ============================================
# RANDOM FOREST EVALUATION
# ============================================

# Predictions
rf_pred = random_forest_pipeline.predict(X_test)
rf_pred_proba = random_forest_pipeline.predict_proba(X_test)[:, 1]

# Metrics
rf_accuracy = accuracy_score(y_test, rf_pred)
rf_precision = precision_score(y_test, rf_pred)
rf_recall = recall_score(y_test, rf_pred)
rf_f1 = f1_score(y_test, rf_pred)
rf_roc_auc = roc_auc_score(y_test, rf_pred_proba)

print("Random Forest Performance")
print("=" * 40)
print(f"Accuracy : {rf_accuracy:.4f}")
print(f"Precision: {rf_precision:.4f}")
print(f"Recall   : {rf_recall:.4f}")
print(f"F1-Score : {rf_f1:.4f}")
print(f"ROC-AUC  : {rf_roc_auc:.4f}")

# Confusion Matrix
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, rf_pred))

Random Forest Performance
Accuracy : 0.7726
Precision: 0.5614
Recall   : 0.6604
F1-Score : 0.6069
ROC-AUC  : 0.8208

Confusion Matrix:
[[840 193]
 [127 247]]


In [16]:
import xgboost as xgb

print("XGBoost version:", xgb.__version__)

XGBoost version: 3.4.1


In [18]:
# ============================================
# XGBOOST PIPELINE
# ============================================

from xgboost import XGBClassifier

xgb_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor_scaled),
        ("model", XGBClassifier(
            n_estimators=300,
            max_depth=5,
            learning_rate=0.05,
            subsample=0.8,
            colsample_bytree=0.8,
            random_state=42,
            eval_metric="logloss",
            n_jobs=-1
        ))
    ]
)

xgb_pipeline.fit(X_train, y_train)

print("XGBoost model trained successfully.")

XGBoost model trained successfully.


In [19]:
# ============================================
# XGBOOST EVALUATION
# ============================================

# Predictions
xgb_pred = xgb_pipeline.predict(X_test)
xgb_pred_proba = xgb_pipeline.predict_proba(X_test)[:, 1]

# Metrics
xgb_accuracy = accuracy_score(y_test, xgb_pred)
xgb_precision = precision_score(y_test, xgb_pred)
xgb_recall = recall_score(y_test, xgb_pred)
xgb_f1 = f1_score(y_test, xgb_pred)
xgb_roc_auc = roc_auc_score(y_test, xgb_pred_proba)

print("XGBoost Performance")
print("=" * 40)
print(f"Accuracy : {xgb_accuracy:.4f}")
print(f"Precision: {xgb_precision:.4f}")
print(f"Recall   : {xgb_recall:.4f}")
print(f"F1-Score : {xgb_f1:.4f}")
print(f"ROC-AUC  : {xgb_roc_auc:.4f}")

# Confusion Matrix
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, xgb_pred))

XGBoost Performance
Accuracy : 0.7818
Precision: 0.6050
Recall   : 0.5160
F1-Score : 0.5570
ROC-AUC  : 0.8248

Confusion Matrix:
[[907 126]
 [181 193]]


In [20]:
# ============================================
# 5-FOLD STRATIFIED CROSS-VALIDATION
# ============================================

from sklearn.model_selection import StratifiedKFold, cross_val_score

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

models = {
    "Logistic Regression": logistic_pipeline,
    "Random Forest": random_forest_pipeline,
    "XGBoost": xgb_pipeline
}

cv_results = {}

for name, model in models.items():
    scores = cross_val_score(
        model,
        X_train,
        y_train,
        cv=cv,
        scoring="roc_auc",
        n_jobs=-1
    )

    cv_results[name] = scores

    print(f"\n{name}")
    print("-" * 40)
    print("Fold ROC-AUC scores:", scores.round(4))
    print(f"Mean ROC-AUC: {scores.mean():.4f}")
    print(f"Std ROC-AUC : {scores.std():.4f}")

/opt/anaconda3/envs/jupyter-ds/lib/python3.12/multiprocessing/queues.py:122: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  return _ForkingPickler.loads(res)
/opt/anaconda3/envs/jupyter-ds/lib/python3.12/multiprocessing/queues.py:122: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  return _ForkingPickler.loads(res)
/opt/anaconda3/envs/jupyter-ds/lib/python3.12/multiprocessing/queues.py:122: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this pack


Logistic Regression
----------------------------------------
Fold ROC-AUC scores: [0.8462 0.8438 0.8375 0.8489 0.8532]
Mean ROC-AUC: 0.8459
Std ROC-AUC : 0.0053


/opt/anaconda3/envs/jupyter-ds/lib/python3.12/multiprocessing/queues.py:122: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  return _ForkingPickler.loads(res)



Random Forest
----------------------------------------
Fold ROC-AUC scores: [0.8298 0.8178 0.8275 0.8364 0.8384]
Mean ROC-AUC: 0.8300
Std ROC-AUC : 0.0073


/opt/anaconda3/envs/jupyter-ds/lib/python3.12/multiprocessing/queues.py:122: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  return _ForkingPickler.loads(res)
/opt/anaconda3/envs/jupyter-ds/lib/python3.12/multiprocessing/queues.py:122: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  return _ForkingPickler.loads(res)



XGBoost
----------------------------------------
Fold ROC-AUC scores: [0.8396 0.8345 0.8415 0.8449 0.8523]
Mean ROC-AUC: 0.8426
Std ROC-AUC : 0.0059


In [21]:
# ============================================
# MODEL EVALUATION & COMPARISON
# ============================================

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

# Generate predictions
logistic_pred = logistic_pipeline.predict(X_test)
logistic_pred_proba = logistic_pipeline.predict_proba(X_test)[:, 1]

rf_pred = random_forest_pipeline.predict(X_test)
rf_pred_proba = random_forest_pipeline.predict_proba(X_test)[:, 1]

xgb_pred = xgb_pipeline.predict(X_test)
xgb_pred_proba = xgb_pipeline.predict_proba(X_test)[:, 1]

# Create comparison table
comparison = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "Random Forest",
        "XGBoost"
    ],

    "Accuracy": [
        accuracy_score(y_test, logistic_pred),
        accuracy_score(y_test, rf_pred),
        accuracy_score(y_test, xgb_pred)
    ],

    "Precision": [
        precision_score(y_test, logistic_pred),
        precision_score(y_test, rf_pred),
        precision_score(y_test, xgb_pred)
    ],

    "Recall": [
        recall_score(y_test, logistic_pred),
        recall_score(y_test, rf_pred),
        recall_score(y_test, xgb_pred)
    ],

    "F1-Score": [
        f1_score(y_test, logistic_pred),
        f1_score(y_test, rf_pred),
        f1_score(y_test, xgb_pred)
    ],

    "ROC-AUC": [
        roc_auc_score(y_test, logistic_pred_proba),
        roc_auc_score(y_test, rf_pred_proba),
        roc_auc_score(y_test, xgb_pred_proba)
    ],

    "CV ROC-AUC": [
        cv_results["Logistic Regression"].mean(),
        cv_results["Random Forest"].mean(),
        cv_results["XGBoost"].mean()
    ]
})

display(comparison.round(4))

,Model,Accuracy,Precision,Recall,F1-Score,ROC-AUC,CV ROC-AUC
0,Logistic Regression,0.8031,0.6474,0.5695,0.6060,0.8362,0.8459
1,Random Forest,0.7726,0.5614,0.6604,0.6069,0.8208,0.8300
2,XGBoost,0.7818,0.6050,0.5160,0.5570,0.8248,0.8426


In [22]:
# ============================================
# SAVE TRAINED MODELS
# ============================================

import joblib
import os

os.makedirs("../models", exist_ok=True)

joblib.dump(
    logistic_pipeline,
    "../models/logistic_regression.pkl"
)

joblib.dump(
    random_forest_pipeline,
    "../models/random_forest.pkl"
)

joblib.dump(
    xgb_pipeline,
    "../models/xgboost.pkl"
)

print("All models saved successfully.")

All models saved successfully.
